# 🎬 100% Free Autonomous AI Video Server (Wan 2.1 / LTX-Video)
### Generates Pixar/Veo-Quality 3D Character & Dinosaur Animations for Free
This notebook runs on Google Colab's **Free T4 GPU** and exposes an API endpoint to connect with your local **Autonomous AI Cartoon Studio** (`E:\YT`).

**Instructions:**
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime > Run all**.
3. Copy the public **`gradio.live`** URL printed at the bottom.
4. Paste it into your local `E:\YT\.env` file as: `WAN21_API_URL=https://xxxx.gradio.live`.

In [ ]:
# Step 1: Install High-Performance Diffusion & Video Generation Libraries
!pip install -q diffusers transformers accelerate torch torchvision torchaudio gradio imageio-ffmpeg

In [ ]:
# Step 2: Load Open-Source Cinematic Video Diffusion Pipeline
import torch
from diffusers import DiffusionPipeline
import gradio as gr
import tempfile
import os

print("Loading open-source cinematic video diffusion pipeline on Free T4 GPU...")
# Using LTX-Video / Wan 2.1 lightweight open pipeline optimized for 15GB T4 VRAM
model_id = "Lightricks/LTX-Video"
pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.to("cuda")
pipe.enable_attention_slicing()
print("✔ Pipeline loaded successfully on GPU!")

In [ ]:
# Step 3: Define Generation Function & Launch Public API Tunnel
from diffusers.utils import export_to_video

def generate_video(prompt: str, negative_prompt: str = "blurry, low quality, distorted", num_frames: int = 49):
    print(f"Generating video for prompt: {prompt[:80]}...")
    with torch.inference_mode():
        video_frames = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_frames=num_frames,
            guidance_scale=7.5
        ).frames[0]
    
    out_path = tempfile.mktemp(suffix=".mp4")
    export_to_video(video_frames, out_path, fps=24)
    print(f"Video saved: {out_path}")
    return out_path

demo = gr.Interface(
    fn=generate_video,
    inputs=[gr.Textbox(label="Prompt"), gr.Textbox(label="Negative Prompt", value="blurry, ugly, distorted")],
    outputs=gr.Video(label="Generated 3D Animation"),
    title="Autonomous AI Cartoon Studio - Free Video Backend"
)

# Share=True creates the free public API tunnel URL for E:\YT
demo.launch(share=True)